# Firestore Collection Explorer

Sample records from the `extracted` and `analysis` collections to understand their structure, key presence, and value distributions.

In [ ]:
import json
import os

import pandas as pd
from dotenv import load_dotenv
from google.cloud import firestore

load_dotenv()

if os.path.isfile("vk-linkedin-master-service-account.json"):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "vk-linkedin-master-service-account.json"

db = firestore.Client(project="vk-linkedin", database="linkedin")
print("Firestore connected")

## 1. Sample `extracted` collection — first 10 records

In [ ]:
SAMPLE_SIZE = 100

extracted_docs = [doc.to_dict() for doc in db.collection("extracted").limit(SAMPLE_SIZE).stream()]
print(f"Fetched {len(extracted_docs)} docs from 'extracted'")


In [ ]:
# Pretty-print the first record in full
print(json.dumps(extracted_docs[0], indent=2, default=str))

## 2. Key frequency in `extracted` — how often each top-level key appears (100 docs)

In [ ]:
FREQ_SAMPLE = 100

freq_docs = [doc.to_dict() for doc in db.collection("extracted").limit(FREQ_SAMPLE).stream()]
total = len(freq_docs)
print(f"Fetched {total} docs for key-frequency analysis")

from collections import Counter

key_counts = Counter()
for doc in freq_docs:
    key_counts.update(doc.keys())

freq_df = pd.DataFrame(
    [(k, v, f"{100*v/total:.1f}%") for k, v in key_counts.most_common()],
    columns=["key", "count", "presence"],
)
freq_df

## 3. Nested key inspection — sub-keys inside complex fields

In [ ]:
NESTED_KEYS = ["miniProfile", "currentPosition", "memberDistance", "externalIds", "positions", "email", "skills", "educations"]

sample = extracted_docs[0]  # use first record from earlier fetch

for key in NESTED_KEYS:
    val = sample.get(key)
    if val is None:
        print(f"\n[{key}] — NOT PRESENT in this record")
        continue
    snippet = json.dumps(val, indent=2, default=str)
    # Truncate long outputs for readability
    if len(snippet) > 800:
        snippet = snippet[:800] + "\n  ... (truncated)"
    print(f"\n[{key}]\n{snippet}")

In [ ]:
# Sub-key frequency across 100 docs for each dict-type nested field
DICT_NESTED_KEYS = ["miniProfile", "currentPosition", "memberDistance", "email"]

for key in DICT_NESTED_KEYS:
    sub_counts = Counter()
    key_present = 0
    for doc in freq_docs:
        val = doc.get(key)
        if isinstance(val, dict):
            sub_counts.update(val.keys())
            key_present += 1
    if key_present == 0:
        print(f"\n[{key}] — not a dict in any sampled doc")
        continue
    rows = [(sk, cnt, f"{100*cnt/key_present:.0f}%") for sk, cnt in sub_counts.most_common()]
    sub_df = pd.DataFrame(rows, columns=["sub-key", "count", "presence"])
    print(f"\n[{key}]  (present in {key_present}/{total} docs)")
    print(sub_df.to_string(index=False))

## 4. Sample `analysis` collection — keys and value distributions

In [ ]:
analysis_docs = [doc.to_dict() for doc in db.collection("analysis").limit(FREQ_SAMPLE).stream()]
total_a = len(analysis_docs)
print(f"Fetched {total_a} docs from 'analysis'")

# Key frequency
a_key_counts = Counter()
for doc in analysis_docs:
    a_key_counts.update(doc.keys())

a_freq_df = pd.DataFrame(
    [(k, v, f"{100*v/total_a:.1f}%") for k, v in a_key_counts.most_common()],
    columns=["key", "count", "presence"],
)
a_freq_df

In [ ]:
# Value distributions for categorical fields
analysis_df = pd.DataFrame(analysis_docs)

for col in ["industry", "function", "seniority"]:
    if col not in analysis_df.columns:
        print(f"\n[{col}] — column not present")
        continue
    counts = analysis_df[col].value_counts(dropna=False)
    print(f"\n--- {col} ({counts.sum()} docs, {analysis_df[col].isna().sum()} missing) ---")
    print(counts.to_string())

In [ ]:
# Show a preview of the analysis DataFrame (first 10 rows, drop long text columns)
preview_cols = [c for c in analysis_df.columns if c != "summary"]
analysis_df[preview_cols].head(10)